In [ ]:
# Imports
import matplotlib.pyplot as plt
from pathlib import Path
from transcription import *
from post_processing import relabel_transcript

In [ ]:
# Select and prepare the audio file
audio_path = Path(r"C:\Users\Somlab\Downloads\BB_VoiceTranscription\audio1381326012.m4a")
audio_duration = prepare_audio(audio_path)
print(f"Audio duration = {audio_duration:0.0f} seconds")

In [ ]:
# Transcribe the audio file
whisper_size = "small"
segments = transcribe(audio_path, whisper_size)
# Print segments
for s in segments:
    print(f"{s.start:0.1f}s - {s.end:0.1f}s: {s.text}")

In [ ]:
# Diarization
diar_model = SortformerEncLabelModel.from_pretrained("nvidia/diar_sortformer_4spk-v1").eval() # type: ignore
speaker_times, needs_post_hoc = diarize_audio(diar_model,
                                              audio_path.with_suffix(".16k.wav"),
                                              segments,
                                              audio_duration)

# Create the transcript from segments and speaker times
transcript = create_transcript(segments, speaker_times)
for t in transcript:
    print(f"{t['start']:0.1f} - {t['end']:0.1f} ({t['speaker']}): {t['text']}")

In [ ]:
# Manual diarization via embedding
# Get speaker start and stop times for extended 
speakers, speaker_start_times, speaker_stop_times = get_transcript_speakers(transcript)

# Get an encoding model for feature extraction
enc_model = EncDecSpeakerLabelModel.from_pretrained(model_name="titanet_small").eval() # type: ignore
device = next(enc_model.parameters()).device
# load the whole audio file
sample_frequency = 16000
audio, _ = librosa.load(audio_path, sr=sample_frequency)

# Get embeddings
embeddings = extract_speaker_segment_embeddings(audio, enc_model, speaker_start_times, speaker_stop_times, device, sample_frequency)
# Compute similarity and classify
similarity_mat = get_pairwise_similarity(embeddings)
distance_mat = 1-similarity_mat
plt.imshow(similarity_mat)

In [ ]:
embedding_matrix = np.vstack(embeddings)
embedding_matrix = embedding_matrix - np.mean(embedding_matrix, axis=0, keepdims=True)
norms = np.linalg.norm(embedding_matrix, axis=1, keepdims=True)
norms[norms == 0] = 1e-12 
embedding_matrix = embedding_matrix / norms
cosine_sim_matrix = np.dot(embedding_matrix, embedding_matrix.T)
plt.imshow(cosine_sim_matrix)

In [ ]:
# Cluster to get harmonized speaker labels
speaker_labels = cluster_embeddings(1-cosine_sim_matrix, 2)
print(speaker_labels)